[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C68_Eval_Infrastructure_Course/02_runner/02_runner.ipynb)

# 02 · Runner 工程（缓存键 / 失败分类 / 重试与偏倚 / 限流 / 预算熔断 / 续跑）

目标：把 runner 从「一个 for 循环」变成一个**不会骗你、不会烧穿预算、崩了能续**的执行器。

本 notebook 你会亲手实现：
1. **缓存键设计** —— 复现「改了 prompt 却读到旧结果」这个不报错的 bug，再修好它
2. **失败分类器** —— 五类失败，各自的重试策略与分母归属
3. **指数退避 + 抖动** —— 以及不加抖动时的惊群效应
4. **重试引入的选择偏倚** —— 「重试到成功为止」会把分数抬高多少
5. **令牌桶限流 + 并发上限** —— 两者管的不是一件事
6. **预算熔断与预检** —— 把「事故」变成「预估」
7. **run summary 的五个数** —— 尤其是「首次运行缓存命中率非 0」这个告警

> 心智模型：**runner 的正确性不是「跑完了」，而是「跑出来的数字没有被执行过程污染」。
> 缓存、重试、并发这三件事，每一件都能在不报错的情况下改变你的分数。**

## 0 · 环境与被评「模型」

In [ ]:
import os, json, math, time, random, hashlib, shutil, itertools
from collections import Counter, defaultdict

import numpy as np

TMP = os.path.abspath('./_eval_tmp')
if os.path.exists(TMP):
    shutil.rmtree(TMP)
os.makedirs(TMP, exist_ok=True)

def stable_hash(obj, n=8):
    payload = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:n]

class RateLimited(Exception): pass
class Transient(Exception): pass
class Timeout(Exception): pass
class Refusal(Exception): pass
class ParseError(Exception): pass

class FakeAPI:
    """可控的被评模型。所有失败都是显式注入的，便于精确验证 runner 的行为。"""

    def __init__(self, prompt_version='v1', accuracy=0.7, seed=0,
                 p_rate_limited=0.0, p_transient=0.0, p_timeout=0.0,
                 p_refusal=0.0, p_parse=0.0, cost_per_call=0.01):
        self.prompt_version = prompt_version
        self.accuracy = accuracy
        self.rng = random.Random(seed)
        self.ps = dict(rate=p_rate_limited, trans=p_transient, to=p_timeout,
                       refuse=p_refusal, parse=p_parse)
        self.cost_per_call = cost_per_call
        self.n_calls = 0
        self.total_cost = 0.0

    def __call__(self, task):
        self.n_calls += 1
        self.total_cost += self.cost_per_call
        r = self.rng.random()
        if r < self.ps['rate']:
            raise RateLimited('429')
        if r < self.ps['rate'] + self.ps['trans']:
            raise Transient('502')
        if r < self.ps['rate'] + self.ps['trans'] + self.ps['to']:
            raise Timeout('deadline exceeded')
        if r < sum([self.ps['rate'], self.ps['trans'], self.ps['to'], self.ps['refuse']]):
            raise Refusal('content policy')
        if r < sum(self.ps.values()):
            raise ParseError('not valid json')
        # prompt_v2 比 v1 强 15 个点 —— 这就是第 1 节要检测的那个改动
        acc = self.accuracy + (0.15 if self.prompt_version == 'v2' else 0.0)
        return {'answer': task['target'] if self.rng.random() < acc else 'WRONG'}

TASKS = [{'task_id': f't{i:03d}', 'input': f'q{i}', 'target': f'a{i}'} for i in range(120)]
api = FakeAPI(seed=1)
print('一次调用:', api(TASKS[0]), '| 累计成本:', f'${api.total_cost:.3f}')
print('\n✅ 五类失败都可以被精确注入——这比真实 API 更适合验证 runner 的正确性。')

## 1 · 缓存键：复现那个不报错的 bug，再修好它

In [ ]:
class Cache:
    def __init__(self):
        self.store = {}
        self.hits = 0
        self.misses = 0

    def get_or_call(self, key, fn):
        if key in self.store:
            self.hits += 1
            return self.store[key], True
        self.misses += 1
        val = fn()
        self.store[key] = val
        return val, False

    @property
    def hit_rate(self):
        n = self.hits + self.misses
        return self.hits / n if n else 0.0


def exec_fingerprint(spec):
    """模块 01 的执行指纹：整个执行配置的哈希。加了新字段它自动就变。"""
    return stable_hash({'model': spec['model'], 'budget': spec['budget'],
                        'prompt_sha': spec['prompt_sha']}, 8)

def key_bad(task, spec):
    return stable_hash(task['input'], 16)                       # ❌ 只有输入内容

def key_good(task, spec):
    return stable_hash([task['input'], exec_fingerprint(spec)], 16)   # ✓ 内容 + 配置指纹


SPEC_V1 = {'model': {'id': 'm1', 'temperature': 0.0},
           'budget': {'retries': 2, 'timeout_s': 30},
           'prompt_sha': stable_hash('PROMPT TEMPLATE v1', 8)}
SPEC_V2 = json.loads(json.dumps(SPEC_V1))
SPEC_V2['prompt_sha'] = stable_hash('PROMPT TEMPLATE v2 —— 改了一句措辞', 8)

def run_with_cache(spec, tasks, cache, key_fn, prompt_version, seed=1):
    api = FakeAPI(prompt_version=prompt_version, accuracy=0.7, seed=seed)
    scores = []
    for t in tasks:
        out, _ = cache.get_or_call(key_fn(t, spec), lambda t=t: api(t))
        scores.append(1.0 if out['answer'] == t['target'] else 0.0)
    return float(np.mean(scores)), api.n_calls

print('=== ❌ 缓存键只含输入内容 ===')
c_bad = Cache()
s1, n1 = run_with_cache(SPEC_V1, TASKS, c_bad, key_bad, 'v1')
s2, n2 = run_with_cache(SPEC_V2, TASKS, c_bad, key_bad, 'v2')   # 改了 prompt
print(f'  prompt v1: 分数 {s1:.1%}  实际调用 {n1} 次')
print(f'  prompt v2: 分数 {s2:.1%}  实际调用 {n2} 次  ← 一次都没调用！')
assert s1 == s2 and n2 == 0
print('  症状：「我明明改了 prompt，分数却一点没变」 → 被误读成「这个改动没有效果」')

print('\n=== ✓ 缓存键 = 内容 + 执行指纹 ===')
c_good = Cache()
s3, n3 = run_with_cache(SPEC_V1, TASKS, c_good, key_good, 'v1')
s4, n4 = run_with_cache(SPEC_V2, TASKS, c_good, key_good, 'v2')
print(f'  prompt v1: 分数 {s3:.1%}  实际调用 {n3} 次')
print(f'  prompt v2: 分数 {s4:.1%}  实际调用 {n4} 次')
assert n4 == len(TASKS) and s4 > s3 + 0.05
print(f'  真实效果：prompt v2 比 v1 高 {s4-s3:.1%} —— 这个改动其实很有效')

# 同配置重跑仍然能省钱
c2 = Cache()
run_with_cache(SPEC_V1, TASKS, c2, key_good, 'v1')
run_with_cache(SPEC_V1, TASKS, c2, key_good, 'v1')
print(f'\n同配置重跑的缓存命中率: {c2.hit_rate:.0%}')
assert c2.hit_rate >= 0.5
print('✅ 缓存该省的钱一分没少省，该失效的时候准确失效——关键是键里用了 exec_fingerprint，')
print('   而不是手写一份「要包含哪些字段」的列表（那种列表加新字段时没人会记得更新）。')

## 2 · 失败分类：五类失败，五种处理

In [ ]:
FAILURE_POLICY = {
    'RateLimited': dict(retry=True,  in_denominator=True,  counts_as='retryable'),
    'Transient':   dict(retry=True,  in_denominator=True,  counts_as='retryable'),
    'Timeout':     dict(retry=True,  in_denominator=True,  counts_as='failure'),
    'Refusal':     dict(retry=False, in_denominator=True,  counts_as='refusal'),
    'ParseError':  dict(retry=True,  in_denominator=True,  counts_as='parse_error'),
    'ScorerError': dict(retry=False, in_denominator=False, counts_as='scorer_error'),
}

def classify(exc):
    return type(exc).__name__

for name, pol in FAILURE_POLICY.items():
    print(f'{name:<14} 重试={str(pol["retry"]):<6} 计入分母={str(pol["in_denominator"]):<6} '
          f'归类={pol["counts_as"]}')

assert FAILURE_POLICY['Refusal']['retry'] is False, '拒答不该重试'
assert FAILURE_POLICY['Timeout']['in_denominator'] is True, '超时必须计入分母'
assert FAILURE_POLICY['ScorerError']['in_denominator'] is False, '判分器崩溃是唯一可排除的'
print('\n✅ 注意 Timeout：它重试，但**如果重试后仍然超时就算失败并计入分母**——')
print('   因为超时是混合信号，一部分来自基础设施，一部分是真实的能力信号。')
print('   把它当成「基础设施错误并重试到成功」，等于系统性地掩盖后者。')

## 3 · 指数退避 + 抖动：不加抖动会怎样

In [ ]:
def backoff_delay(attempt, base=0.1, cap=8.0, jitter=True, rng=None):
    d = min(base * (2 ** attempt), cap)
    if jitter:
        rng = rng or random
        d *= rng.uniform(0.5, 1.5)
    return d

print(f"{'第几次重试':>10}{'无抖动':>10}{'有抖动(样本)':>16}")
rng = random.Random(0)
for k in range(6):
    no_j = backoff_delay(k, jitter=False)
    with_j = [round(backoff_delay(k, jitter=True, rng=rng), 3) for _ in range(3)]
    print(f'{k:>10}{no_j:>10.2f}{str(with_j):>16}')

# 惊群效应：100 个同时被限流的请求，在同一时刻重试
def herd_spread(n=200, attempt=3, jitter=True, seed=0):
    rng = random.Random(seed)
    times = [backoff_delay(attempt, jitter=jitter, rng=rng) for _ in range(n)]
    buckets = Counter(round(t, 1) for t in times)
    return max(buckets.values()), len(buckets)

peak_no, slots_no = herd_spread(jitter=False)
peak_yes, slots_yes = herd_spread(jitter=True)
print(f'\n200 个同时被限流的请求，第 3 次重试时:')
print(f'  无抖动: 全部挤在 {slots_no} 个时刻，峰值 {peak_no} 个请求  ← 把上游再打垮一次')
print(f'  有抖动: 摊到 {slots_yes} 个时刻，峰值 {peak_yes} 个请求')
assert peak_no == 200 and peak_yes < 40
print('\n✅ 抖动那一项不是可选的——没有它，重试本身就是下一次雪崩的原因。')

## 4 · 重试引入的选择偏倚：「重试到成功为止」抬高多少分

In [ ]:
def run_with_retry_policy(tasks, policy, seed=0, max_retries=3, true_acc=0.55):
    """policy: 'infra_only'（只对基础设施错误重试，正确）
               'until_pass'（判分为 0 就重试，严重偏倚）
               'on_parse'  （解析失败也重试，轻微偏倚）"""
    rng = random.Random(seed)
    scores, n_calls = [], 0

    def one_call():
        nonlocal n_calls
        n_calls += 1
        r = rng.random()
        if r < 0.08:
            raise Transient('502')
        if r < 0.14:
            raise ParseError('bad json')
        return 1.0 if rng.random() < true_acc else 0.0

    for _ in tasks:
        score = None
        for attempt in range(max_retries + 1):
            try:
                score = one_call()
            except Transient:
                continue                                  # 所有策略都重试
            except ParseError:
                if policy in ('on_parse', 'until_pass'):
                    continue
                score = 0.0                               # 不重试就记为失败
                break
            # 拿到判分结果之后还重试 —— 这就是偏倚的来源
            if policy == 'until_pass' and score == 0.0 and attempt < max_retries:
                continue
            break
        scores.append(0.0 if score is None else score)
    return float(np.mean(scores)), n_calls

TRUE_ACC = 0.55
print(f'真实能力: {TRUE_ACC:.0%}\n')
print(f"{'重试策略':<24}{'观测分数':>10}{'偏离真值':>12}{'调用次数':>10}")
res = {}
for pol, label in [('infra_only', '只重试基础设施错误 ✓'),
                   ('on_parse', '解析失败也重试'),
                   ('until_pass', '判分为 0 就重试 ❌')]:
    s, n = run_with_retry_policy(TASKS, pol, seed=7, true_acc=TRUE_ACC)
    res[pol] = s
    print(f'{label:<24}{s:>10.1%}{s - TRUE_ACC:>+12.1%}{n:>10}')

assert abs(res['infra_only'] - TRUE_ACC) < 0.06, '正确策略应当接近真值'
assert res['until_pass'] > res['infra_only'] + 0.15, '「重试到成功」必然大幅抬高分数'
print(f'\n✅ 「判分为 0 就重试」把 {TRUE_ACC:.0%} 的真实能力抬到了 {res["until_pass"]:.0%}。')
print('   它不会报错、日志看起来正常、代码读起来也很合理——')
print('   **这就是为什么规则必须写死：重试的判断只能看异常类型，不能看判分结果。**')
print('   代码结构上的落实方式：重试逻辑必须在判分之前完成。')

## 5 · 令牌桶限流 + 并发上限：两者管的不是一件事

In [ ]:
class TokenBucket:
    """令牌桶：平时允许突发，长期严格遵守平均速率。"""

    def __init__(self, rate, burst, now=0.0):
        self.rate = rate
        self.burst = burst
        self.tokens = float(burst)
        self.t = now
        self.total_wait = 0.0

    def acquire(self, now, n=1):
        """返回需要等待的时间（模拟时钟，不真的 sleep）。"""
        self.tokens = min(self.burst, self.tokens + (now - self.t) * self.rate)
        self.t = now
        if self.tokens >= n:
            self.tokens -= n
            return 0.0
        need = (n - self.tokens) / self.rate
        self.tokens = 0.0
        self.t = now + need
        self.total_wait += need
        return need

def simulate(n_req, rate, burst, concurrency, service_time=0.05):
    """模拟时钟下的执行：并发上限决定同时在飞多少，限流器决定发出速率。"""
    bucket = TokenBucket(rate, burst)
    now, in_flight, finished, wait_total = 0.0, [], 0, 0.0
    for i in range(n_req):
        while len(in_flight) >= concurrency:
            now = max(now, min(in_flight))
            in_flight = [t for t in in_flight if t > now]
            finished += 1
        w = bucket.acquire(now)
        wait_total += w
        now += w
        in_flight.append(now + service_time)
    makespan = max(in_flight) if in_flight else now
    return {'makespan': makespan, 'throughput': n_req / makespan,
            'wait_ratio': wait_total / makespan if makespan else 0.0}

print(f"{'并发':>6}{'限流(req/s)':>14}{'总耗时(s)':>12}{'吞吐(req/s)':>14}{'限流等待占比':>14}")
for conc in [1, 4, 8, 16, 32]:
    r = simulate(400, rate=20, burst=20, concurrency=conc)
    print(f'{conc:>6}{20:>14}{r["makespan"]:>12.1f}{r["throughput"]:>14.1f}'
          f'{r["wait_ratio"]:>14.0%}')

r4 = simulate(400, rate=20, burst=20, concurrency=4)
r32 = simulate(400, rate=20, burst=20, concurrency=32)
assert abs(r32['throughput'] - r4['throughput']) < 3, '限流封顶后，加并发不再提高吞吐'
assert r32['wait_ratio'] > r4['wait_ratio'], '加并发只是让更多请求在等令牌'
print('\n✅ 限流器把吞吐封在 20 req/s 之后，把并发从 4 加到 32 完全没用——')
print('   只是让更多请求在排队等令牌（等待占比从 '
      f'{r4["wait_ratio"]:.0%} 涨到 {r32["wait_ratio"]:.0%}）。')
print('   **限流等待占比 > 30% 就是「并发设得比配额高」的信号**——加并发纯属浪费。')

## 6 · 预算保护：预检 + 熔断

In [ ]:
def dry_run_estimate(tasks, cost_per_call, n_probe=10, attempts=1):
    """先跑一小批，外推总成本。这一步能抓住绝大多数「配置写错导致成本爆炸」。"""
    probe_cost = n_probe * attempts * cost_per_call
    per_task = probe_cost / n_probe
    return {'n_probe': n_probe, 'per_task': per_task,
            'estimated_total': per_task * len(tasks)}

est = dry_run_estimate(TASKS, cost_per_call=0.01, attempts=3)
print(f'预检: {est["n_probe"]} 条 → 单条 ${est["per_task"]:.4f} → '
      f'全量 {len(TASKS)} 条预计 ${est["estimated_total"]:.2f}')

# 配置写错的情况：attempts 被误写成 50
est_bad = dry_run_estimate(TASKS, cost_per_call=0.01, attempts=50)
print(f'如果 attempts 误写成 50: 预计 ${est_bad["estimated_total"]:.2f}  '
      f'← 预检会在花钱之前就把它暴露出来')
assert est_bad['estimated_total'] > 10 * est['estimated_total']
print('\n✅ 预检便宜到几乎没有理由不做：跑 10 条，打印预估总花费，然后再继续。')

In [ ]:
class CircuitBreaker:
    """三状态熔断器。half_open 是关键——它让熔断能**自动恢复**，无人值守时尤其重要。"""

    def __init__(self, window=20, threshold=0.5, cooldown=5.0):
        self.window, self.threshold, self.cooldown = window, threshold, cooldown
        self.recent = []
        self.state = 'closed'
        self.opened_at = None

    def allow(self, now):
        if self.state == 'open':
            if now - self.opened_at >= self.cooldown:
                self.state = 'half_open'
                return True                       # 放一个请求过去试探
            return False
        return True

    def record(self, ok, now):
        if self.state == 'half_open':
            self.state = 'closed' if ok else 'open'
            if not ok:
                self.opened_at = now
                self.cooldown *= 2                # 失败则延长等待
            self.recent = []
            return
        self.recent.append(ok)
        if len(self.recent) > self.window:
            self.recent.pop(0)
        if len(self.recent) == self.window and (1 - np.mean(self.recent)) > self.threshold:
            self.state = 'open'
            self.opened_at = now

cb = CircuitBreaker(window=10, threshold=0.5, cooldown=3.0)
timeline = []
now = 0.0
# 前 10 步正常，第 10-39 步上游挂了（全部失败），第 40 步起恢复
for step in range(80):
    now += 0.5
    allowed = cb.allow(now)
    state_at_allow = cb.state          # ← half_open 是**瞬态**：allow 里进入，record 里立刻离开
    if allowed:
        ok = (step < 10) or (step >= 40)
        cb.record(ok, now)
    timeline.append((step, state_at_allow, allowed, cb.state))

probe_steps = [i for i, s, _, _ in timeline if s == 'half_open']
blocked = sum(1 for _, _, a, _ in timeline if not a)
transitions = [f'{i}:{after}' for i, _, _, after in timeline
               if i == 0 or after != timeline[i-1][3]]
print('状态变化:', transitions)
print(f'进入 half_open 试探的时刻: {probe_steps}')
print(f'被熔断挡下的请求: {blocked} / 80')
assert 'open' in [t[3] for t in timeline], '上游全挂时必须熔断'
assert len(probe_steps) >= 2, '必须反复进入试探状态'
assert timeline[-1][3] == 'closed', '上游恢复后必须自动回到正常'
assert blocked > 20
print('\n✅ 熔断 → 试探（失败则延长等待）→ 再试探 → 恢复，全程无人介入。')
print('   注意 half_open 是一个**瞬态**：allow() 里进入，record() 里立刻转向 closed 或 open，')
print('   所以只有在 allow 的那一刻才观察得到——这也是它只放行一个请求的实现方式。')
print('   没有它的话，熔断之后需要人工重启，而评测经常在夜间/CI 里无人值守地跑。')

## 7 · Run Summary：五个必报的数

In [ ]:
def run_summary(run_id, rows, cache, n_retries, concurrency, wait_ratio,
                cost, cost_estimate):
    kinds = Counter(r['kind'] for r in rows)
    n = len(rows)
    n_scorer_err = kinds.get('scorer_error', 0)
    denom = n - n_scorer_err
    ok = kinds.get('ok', 0)
    return {
        'run_id': run_id,
        'n_rows': n,
        'breakdown': dict(kinds),
        'score': ok / denom if denom else float('nan'),
        'cache_hit_rate': cache.hit_rate,
        'retry_rate': n_retries / n if n else 0.0,
        'concurrency': concurrency,
        'ratelimit_wait_ratio': wait_ratio,
        'cost_usd': cost,
        'cost_vs_estimate': (cost - cost_estimate) / cost_estimate if cost_estimate else 0.0,
        'scorer_error_rate': n_scorer_err / n if n else 0.0,
    }

rng = np.random.default_rng(3)
KINDS = ['ok'] * 1489 + ['timeout'] * 22 + ['refusal'] * 11 + \
        ['parse_error'] * 9 + ['scorer_error'] * 5
rows = [{'kind': k} for k in KINDS]
fresh_cache = Cache(); fresh_cache.misses = len(rows)
summ = run_summary('f2a91b7c-20260829-1430', rows, fresh_cache,
                   n_retries=49, concurrency=8, wait_ratio=0.114,
                   cost=12.47, cost_estimate=11.80)
print(f'RUN SUMMARY · {summ["run_id"]}')
print(f'  行数 {summ["n_rows"]} | 分解 {summ["breakdown"]}')
print(f'  分数 {summ["score"]:.1%}   （分母已排除 {summ["breakdown"].get("scorer_error",0)} 条判分器崩溃）')
for k in ['cache_hit_rate', 'retry_rate', 'ratelimit_wait_ratio',
          'cost_vs_estimate', 'scorer_error_rate']:
    print(f'  {k:<24} {summ[k]:.2%}')

HEALTH = {
    'cache_hit_rate':      lambda v, first: (v == 0.0) if first else (v > 0.9),
    'retry_rate':          lambda v, first: v < 0.05,
    'ratelimit_wait_ratio':lambda v, first: v < 0.30,
    'cost_vs_estimate':    lambda v, first: abs(v) < 0.20,
    'scorer_error_rate':   lambda v, first: v < 0.01,
}
alerts = [k for k, fn in HEALTH.items() if not fn(summ[k], True)]
print(f'\n健康检查未通过的项: {alerts}')
assert alerts == [], '本次运行应当全部健康'

# 制造一个「首次运行却有缓存命中」的情况 —— run_id 被意外复用
reused = Cache(); reused.hits = 300; reused.misses = 1236
summ2 = run_summary('same-run-id', rows, reused, 49, 8, 0.114, 12.47, 11.80)
alerts2 = [k for k, fn in HEALTH.items() if not fn(summ2[k], True)]
print(f'run_id 被复用时的告警: {alerts2}  (命中率 {summ2["cache_hit_rate"]:.0%})')
assert 'cache_hit_rate' in alerts2
print('\n✅ 「首次运行的缓存命中率非 0」是一个高价值告警——')
print('   它几乎总是意味着 run_id 被复用，而复用会让新旧配置的结果混进同一个 run。')
print('   这类污染连指纹校验都抓不到（写入用当前指纹，混进去的旧行用旧指纹），')
print('   所以聚合时那条 assert「一个 run 内指纹唯一」是必须的。')

## ✏️ 练习 1：缓存键的完整性检查

实现 `cache_key_covers(key_fn, spec, task, mutations)`：
对 `mutations`（一个 `{描述: 修改后的 spec}` 字典）逐个检查
`key_fn(task, mutated_spec) != key_fn(task, spec)`，
返回 `(是否全部覆盖, 未被覆盖的修改描述列表)`。

In [ ]:
def cache_key_covers(key_fn, spec, task, mutations):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
def mut(spec, path, value):
    s = json.loads(json.dumps(spec))
    d = s
    for k in path[:-1]:
        d = d[k]
    d[path[-1]] = value
    return s

MUTATIONS = {
    '改 prompt':      mut(SPEC_V1, ['prompt_sha'], 'deadbeef'),
    '改模型':         mut(SPEC_V1, ['model', 'id'], 'm2'),
    '改温度':         mut(SPEC_V1, ['model', 'temperature'], 0.7),
    '改重试次数':     mut(SPEC_V1, ['budget', 'retries'], 5),
}
ok_bad, miss_bad = cache_key_covers(key_bad, SPEC_V1, TASKS[0], MUTATIONS)
ok_good, miss_good = cache_key_covers(key_good, SPEC_V1, TASKS[0], MUTATIONS)
print(f'❌ key_bad : 全覆盖={ok_bad}, 漏掉 {len(miss_bad)} 项 → {miss_bad}')
print(f'✓ key_good: 全覆盖={ok_good}, 漏掉 {len(miss_good)} 项')
assert ok_bad is False and len(miss_bad) == len(MUTATIONS)
assert ok_good is True and miss_good == []
print('✅ 练习 1 通过：这个检查应当作为单元测试跑在 CI 里——')
print('   每加一个配置项，就往 MUTATIONS 里加一条，确保缓存键真的覆盖了它。')

## ✏️ 练习 2：失败分类的分母计算

实现 `compute_denominators(rows)`：输入 `[{'kind': ...}, ...]`，
返回 `{'n_total', 'n_denominator', 'n_success', 'score', 'excluded'}`，
其中分母排除 `scorer_error`，`n_success` 只算 `kind == 'ok'`。

In [ ]:
def compute_denominators(rows):
    # TODO：用上面的 FAILURE_POLICY 判断哪些计入分母
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
POLICY_KIND = {'ok': True, 'retryable': True, 'failure': True, 'timeout': True,
               'refusal': True, 'parse_error': True, 'scorer_error': False}
r = compute_denominators(rows)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
assert r['n_total'] == len(rows)
assert r['excluded'] == 5 and r['n_denominator'] == len(rows) - 5
assert abs(r['score'] - 1489 / (len(rows) - 5)) < 1e-9

# 全部是判分器崩溃的极端情形
r2 = compute_denominators([{'kind': 'scorer_error'}] * 10)
assert r2['n_denominator'] == 0 and math.isnan(r2['score'])
print('\n全部判分器崩溃时: 分母为 0，分数为 nan（而不是 0 或 1）')
print('✅ 练习 2 通过：分母排除 scorer_error，但它的**比例必须被报告**——')
print('   超过 1% 就说明判分器有 bug，这份结果的可信度存疑。')

## ✏️ 练习 3：重试预算的分配

实现 `retry_budget(n_tasks, base_calls, retry_rate, max_retries)`：
估算总调用次数 = `n_tasks * base_calls * (1 + retry_rate + retry_rate^2 + ... + retry_rate^max_retries)`
（等比级数，因为每次重试也可能再失败）。返回 `(总调用次数, 相对无重试的倍数)`。

In [ ]:
def retry_budget(n_tasks, base_calls, retry_rate, max_retries):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
n0, m0 = retry_budget(500, 1, 0.0, 3)
assert n0 == 500 and abs(m0 - 1.0) < 1e-12
n1, m1 = retry_budget(500, 1, 0.10, 3)
n2, m2 = retry_budget(500, 1, 0.50, 3)
assert n2 > n1 > n0
print(f"{'失败率':>8}{'总调用':>10}{'成本倍数':>10}")
for rr in [0.0, 0.05, 0.10, 0.30, 0.50, 0.80]:
    n_, m_ = retry_budget(500, 1, rr, 3)
    print(f'{rr:>8.0%}{n_:>10.0f}{m_:>10.2f}x')
n_hi, m_hi = retry_budget(500, 1, 0.80, 3)
assert m_hi > 2.5
print('✅ 练习 3 通过：失败率 30% 时成本就涨了四成，80% 时接近 3 倍——')
print('   **预算估算必须把重试算进去**，否则预检的估计会系统性偏低。')

## ✏️ 练习 4：并发的最优选择

实现 `best_concurrency(n_req, rate, burst, candidates, wait_cap=0.30)`：
在候选并发度里，选出「限流等待占比 ≤ wait_cap」且吞吐最高的那个。
返回 `(最优并发, 该并发下的吞吐, 等待占比)`。

In [ ]:
def best_concurrency(n_req, rate, burst, candidates, wait_cap=0.30):
    # TODO：用上面的 simulate
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
CANDS = [1, 2, 4, 8, 16, 32, 64]
c, tp, wr = best_concurrency(400, rate=20, burst=20, candidates=CANDS)
print(f'配额 20 req/s 时的最优并发: {c}  (吞吐 {tp:.1f} req/s, 等待占比 {wr:.0%})')
assert wr <= 0.30
assert c < max(CANDS), '最优并发不该是最大的那个'

c2, tp2, wr2 = best_concurrency(400, rate=100, burst=100, candidates=CANDS)
print(f'配额 100 req/s 时的最优并发: {c2}  (吞吐 {tp2:.1f} req/s)')
assert c2 >= c, '配额更高时，最优并发应当不降低'
print('✅ 练习 4 通过：最优并发由**上游配额**决定，不是越大越好——')
print('   超过配额之后加并发只增加等待，不增加吞吐，还会推高 429 率与超时率。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cache_key_covers(key_fn, spec, task, mutations):
    base = key_fn(task, spec)
    missed = [desc for desc, m in mutations.items() if key_fn(task, m) == base]
    return (len(missed) == 0, missed)

In [ ]:
# 练习 2 参考答案
def compute_denominators(rows):
    kinds = Counter(r['kind'] for r in rows)
    excluded = sum(c for k, c in kinds.items() if not POLICY_KIND.get(k, True))
    n_total = len(rows)
    n_denom = n_total - excluded
    n_succ = kinds.get('ok', 0)
    return {'n_total': n_total, 'n_denominator': n_denom, 'n_success': n_succ,
            'score': (n_succ / n_denom) if n_denom else float('nan'),
            'excluded': excluded}

In [ ]:
# 练习 3 参考答案
def retry_budget(n_tasks, base_calls, retry_rate, max_retries):
    factor = sum(retry_rate ** k for k in range(max_retries + 1))
    total = n_tasks * base_calls * factor
    return (total, factor)

In [ ]:
# 练习 4 参考答案
def best_concurrency(n_req, rate, burst, candidates, wait_cap=0.30):
    best = None
    for c in candidates:
        r = simulate(n_req, rate, burst, c)
        if r['wait_ratio'] <= wait_cap:
            if best is None or r['throughput'] > best[1]:
                best = (c, r['throughput'], r['wait_ratio'])
    if best is None:                       # 全都超过等待上限，退回最小并发
        r = simulate(n_req, rate, burst, min(candidates))
        best = (min(candidates), r['throughput'], r['wait_ratio'])
    return best

---
## 🧪 真实工程胶囊：runner 的落地要点

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 缓存键：永远用执行指纹，不要手写字段列表
# ══════════════════════════════════════════════════════════════════
def cache_key(task, spec):
    return sha256(json.dumps({
        "input": task["input"],
        "exec_fp": exec_fingerprint(spec),   # ← 整个执行配置的哈希，加字段自动生效
        "cache_format": 3,                   # ← 存储结构变更时整体失效
    }, sort_keys=True).encode()).hexdigest()

# 单元测试（练习 1）：每加一个配置项就往 MUTATIONS 里加一条
def test_cache_key_covers_all_config():
    for path, val in [(["model","id"],"other"), (["model","temperature"],0.7),
                      (["prompt_sha"],"x"), (["budget","retries"],9),
                      (["tools"],["a","b"])]:
        assert cache_key(T, mutate(SPEC, path, val)) != cache_key(T, SPEC), path

# ══════════════════════════════════════════════════════════════════
# B. 重试：只看异常类型，绝不看判分结果
# ══════════════════════════════════════════════════════════════════
RETRYABLE = (ConnectionError, TimeoutError, RateLimitError, InternalServerError)

async def call_with_retry(fn, *a, max_retries=3, **kw):
    for k in range(max_retries + 1):
        try:
            return await fn(*a, **kw)
        except RETRYABLE as e:
            if k == max_retries:
                raise
            await asyncio.sleep(min(0.5 * 2**k, 8) * random.uniform(0.5, 1.5))  # 抖动必需
# 结构上的落实：这个函数**拿不到 scorer**，所以它在物理上不可能"重试到通过为止"。

# ══════════════════════════════════════════════════════════════════
# C. 并发 + 限流：两个都要有
# ══════════════════════════════════════════════════════════════════
sem = asyncio.Semaphore(CONCURRENCY)         # 同时在飞多少（保护内存/连接数）
limiter = AsyncLimiter(RATE_PER_MIN, 60)     # 单位时间发多少（遵守上游配额）

async def guarded(task):
    async with sem, limiter:
        return await call_with_retry(api_call, task)

# inspect-ai: --max-connections N 对应 sem；限流通常由 SDK 内置的 retry 处理。
# **报告里必须写 concurrency**（C66-05 复现清单第 7 条）。

# ══════════════════════════════════════════════════════════════════
# D. 上线前必做：并发敏感性检查
# ══════════════════════════════════════════════════════════════════
# for c in [1, 4, 8, 16]:
#     run_eval(spec, smoke_subset, concurrency=c)
# 分数随 c 变化 → 超时阈值设得太紧，这个分数不能用。
# 分数对 c 不敏感 → ✅ 超时有余量，超时反映的是真实能力而非基础设施。

# ══════════════════════════════════════════════════════════════════
# E. 每次运行结束必须打印的五个数（讲解第 7 节）
# ══════════════════════════════════════════════════════════════════
# cache_hit_rate（首次应为 0）· retry_rate（<5%）· concurrency
# ratelimit_wait_ratio（<30%）· cost vs 预检估计（偏差<20%）· scorer_error_rate（→0）
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 缓存键 | 用 `exec_fingerprint` 而不是手写字段列表 | 每个 runner |
| 何时关缓存 | 估方差、算区间、跑 pass^k 时必须关 | 统计分析 |
| 失败五分类 | 只有 `scorer_error` 可以从分母排除，且必须报告比例 | 结果聚合 |
| 退避 + 抖动 | 抖动不是可选的——没有它重试就是下一次雪崩 | 重试实现 |
| 重试的偏倚 | **重试只看异常类型，不看判分结果**；结构上让重试拿不到 scorer | 最重要的一条 |
| 并发 vs 限流 | 两者管的不是一件事，都要有；等待占比 >30% = 并发白加 | 性能与成本 |
| 预检与熔断 | 预检把事故变成预估；half_open 让熔断能自动恢复 | 预算保护 |
| 五个必报的数 | 「首次运行缓存命中率非 0」= run_id 被复用 | run summary |

下一模块：**03 · 结果存储与分析**——schema 设计、切片查询、
以及「为什么聚合逻辑应该和结果分开」。